# Phase 5 v3.1 — Fine-Tuning (All v3 Fixes + Correct Image Handling)

**v2 → v3 changes (evidence-backed):**
1. **Response-only loss masking** — QLoRA paper Table 10: +2-5%.
2. **REMOVED `modules_to_save`** — PEFT #1750/#2864: unties lm_head/embed_tokens.
3. **3 epochs + early stopping** (was 1 epoch / 94 steps). patience=2.
4. **LR 1e-4** (was 2e-4) — preserves pretrained histopathology features.
5. **MSI-H oversampling disabled** — 4 unique patients = memorization, not learning.

**Image processing (fully understood now):**

MedGemma's vision pipeline (from model card + source):
1. Processor resizes images to **896×896** (native SigLIP resolution)
2. SigLIP patch_size=14 → 64×64 = **4096 internal patches** (position embeddings)
3. `Gemma3MultiModalProjector` 4×4 avg-pool → **256 LLM tokens per image**
4. Each image = 256 tokens in the LLM sequence, regardless of input pixel size

**Why resizing to 224×224 crashed:**
- 224/14 = 16×16 = 256 patches, but SigLIP position embedding table has 4096 entries
- `embeddings + position_embedding(position_ids)` → tensor(256) vs tensor(4096) → CRASH
- The processor MUST feed 896×896 to SigLIP. Do NOT override `processor.image_processor.size`.

**OOM fix:**
- batch=8 × 4 imgs = 32 images through SigLIP attention (eager) = 32×16×4096²×4 = 34 GB → OOM
- Fix: batch=2, grad_accum=8 → 8 images through SigLIP = 8.6 GB → safe
- `do_pan_and_scan = False` (default) → no extra crops

**Token budget with max_patches=2:**
- 2 images × 256 tokens + ~300 text = **~812 total LLM tokens**
- `batch=2, grad_accum=8`: effective batch = 16
- SigLIP attention: ~4.3 GB, LLM attention: negligible at seq=812
- Estimated peak VRAM: ~19 GB / 80 GB → safe

**What stayed the same:**
- LoRA r=16, alpha=16, all-linear, QLoRA 4-bit NF4
- SFTTrainer, adamw_torch_fused, cosine scheduler, TF32, image pre-caching

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/ImmunoPath"
DATA_DIR = f"{PROJECT_DIR}/data"
TRAINING_DIR = f"{DATA_DIR}/training_v3"  # V3.1: new training data with schema fixes
MODEL_DIR = f"{PROJECT_DIR}/models/immunopath-v3.1"
RESULTS_DIR = f"{PROJECT_DIR}/results/phase5_v3"

# Local SSD paths — critical for speed
LOCAL_CKPT_DIR = "/content/immunopath_ckpts_v3"
LOCAL_LOG_DIR = "/content/immunopath_logs_v3"

for d in [MODEL_DIR, RESULTS_DIR, LOCAL_CKPT_DIR, LOCAL_LOG_DIR]:
    os.makedirs(d, exist_ok=True)

import subprocess
subprocess.run([
    "pip", "install", "-q", "--no-cache-dir",
    "transformers>=4.50.0",
    "accelerate>=0.34.0",
    "peft>=0.12.0",
    "bitsandbytes>=0.44.0",
    "trl>=0.12.0",
    "datasets",
    "tqdm",
], check=True)

# === Flash Attention 2 — try import first, install with timeout if missing ===
# FlashAttn-2 uses tiled O(N) memory instead of O(N²) for attention.
# On A100, this halves attention VRAM and adds ~20% speed.
# Colab A100 instances often have it pre-installed.
FLASH_ATTN_AVAILABLE = False
try:
    import flash_attn
    FLASH_ATTN_AVAILABLE = True
    print(f"✓ Flash Attention {flash_attn.__version__} already installed")
except ImportError:
    print("Flash Attention not found — attempting install (180s timeout)...")
    try:
        result = subprocess.run(
            ["pip", "install", "-q", "flash-attn", "--no-build-isolation"],
            timeout=1,
            capture_output=True,
            text=True,
        )
        if result.returncode == 0:
            import flash_attn
            FLASH_ATTN_AVAILABLE = True
            print(f"✓ Flash Attention {flash_attn.__version__} installed")
        else:
            print(f"flash-attn install failed: {result.stderr[:200]}")
            print("Continuing with eager attention")
    except subprocess.TimeoutExpired:
        print("flash-attn install timed out — using eager attention (still safe)")
    except Exception as e:
        print(f"flash-attn install error: {e} — using eager attention")

if FLASH_ATTN_AVAILABLE:
    print("  → Attention memory: O(N) instead of O(N²) — enables even larger batches")
else:
    print("  → Eager attention: O(N²) — safe with batch=2 and 256 tokens per image")

from huggingface_hub import login
from google.colab import userdata
try:
    login(token=userdata.get("HF_TOKEN"))
    print("HuggingFace login ✓")
except Exception:
    print("Set HF_TOKEN in Colab Secrets")

import torch
import gc

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nGPU: {gpu_name} ({gpu_mem:.1f} GB)")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    if "A100" in gpu_name:
        flash_status = "FlashAttn-2 ✓" if FLASH_ATTN_AVAILABLE else "eager attn (safe)"
        print(f"  ✓ A100 detected — TF32 enabled, {flash_status}")
else:
    print("No GPU detected!")

import transformers, peft, trl
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print(f"trl: {trl.__version__}")


Mounted at /content/drive
Flash Attention not found — attempting install (180s timeout)...
flash-attn install timed out — using eager attention (still safe)
  → Eager attention: O(N²) — safe with batch=2 and 256 tokens per image
HuggingFace login ✓

GPU: NVIDIA A100-SXM4-80GB (85.1 GB)
  ✓ A100 detected — TF32 enabled, eager attn (safe)
transformers: 5.0.0
peft: 0.18.1
trl: 0.28.0


## Configuration (v3 changes + speed optimizations highlighted)

In [3]:
from dataclasses import dataclass, field

@dataclass
class Config:
    model_id: str = "google/medgemma-1.5-4b-it"

    # LoRA — same as Google's official notebook
    lora_r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.05
    target_modules: str = "all-linear"

    # V3 FIX #2: REMOVED modules_to_save (PEFT #1750/#2864)
    modules_to_save: list = field(default_factory=list)

    # 3 epochs with early stopping (patience=2) — CU budget constraint
    num_epochs: int = 3

    # === BATCH CONFIG ===
    # SigLIP processes each image at 896×896 → 4096 internal patches
    # SigLIP attention (eager, float32 softmax) per batch:
    #   N_images × 16_heads × 4096² × 4 bytes
    #   batch=2, patches=4 → 8 imgs → 8.6 GB (safe)
    #   batch=8, patches=4 → 32 imgs → 34.4 GB → OOM!
    # After projector: each image = 256 LLM tokens (4×4 avg-pool)
    # LLM sequence: 4 × 256 + ~300 text = ~1324 tokens → negligible
    batch_size: int = 2
    grad_accum: int = 8     # effective batch = 16
    eval_batch_size: int = 4  # eval has no gradients → can be larger

    # LR 1e-4 (was 2e-4 — preserves pretrained features)
    lr: float = 1e-4

    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 0.3
    lr_scheduler_type: str = "cosine"

    # MSI-H oversampling disabled (4 unique patients = memorization)
    msi_h_oversample: int = 1

    # max_length=2048: each image = 256 LLM tokens after projector
    # 2 images × 256 tokens + ~300 text = ~812 tokens < 2048
    max_patches: int = 2
    max_length: int = 2048

    # Paths
    train_path: str = f"{TRAINING_DIR}/train.jsonl"
    val_path: str = f"{TRAINING_DIR}/val.jsonl"
    output_dir: str = MODEL_DIR
    log_dir: str = RESULTS_DIR

    logging_steps: int = 10

cfg = Config()

# Print VRAM budget
llm_seq_est = cfg.max_patches * 256 + 300  # LLM sequence length
n_siglip_imgs = cfg.batch_size * cfg.max_patches  # images through SigLIP per step
siglip_attn_gb = n_siglip_imgs * 16 * 4096**2 * 4 / 1e9  # float32 softmax

print(f"Config (v3.1):")
print(f"  LoRA: r={cfg.lora_r}, alpha={cfg.lora_alpha}, target={cfg.target_modules}")
print(f"  modules_to_save: {cfg.modules_to_save} ← EMPTY (PEFT #1750/#2864 fix)")
print(f"  Batch: {cfg.batch_size}×{cfg.grad_accum} = {cfg.batch_size*cfg.grad_accum} effective")
print(f"  Eval batch: {cfg.eval_batch_size}")
print(f"  Images: native res → processor resizes to 896×896 → 256 LLM tokens per image")
print(f"  LLM seq budget: ~{llm_seq_est} tokens")
print(f"  LR: {cfg.lr}, scheduler: {cfg.lr_scheduler_type}")
print(f"  Epochs: {cfg.num_epochs} + early stopping (patience=2)")
print(f"  MSI-H oversample: {cfg.msi_h_oversample}× (disabled)")
print(f"\n  VRAM budget:")
print(f"  SigLIP: {n_siglip_imgs} images × 16 heads × 4096² × 4B = {siglip_attn_gb:.1f} GB")
print(f"  Model (4-bit): ~3 GB")
print(f"  LoRA + optimizer: ~1.5 GB")
print(f"  Estimated peak: ~{siglip_attn_gb + 3 + 1.5 + 10:.0f} GB / 80 GB")


Config (v3.1):
  LoRA: r=16, alpha=16, target=all-linear
  modules_to_save: [] ← EMPTY (PEFT #1750/#2864 fix)
  Batch: 2×8 = 16 effective
  Eval batch: 4
  Images: native res → processor resizes to 896×896 → 256 LLM tokens per image
  LLM seq budget: ~812 tokens
  LR: 0.0001, scheduler: cosine
  Epochs: 3 + early stopping (patience=2)
  MSI-H oversample: 1× (disabled)

  VRAM budget:
  SigLIP: 4 images × 16 heads × 4096² × 4B = 4.3 GB
  Model (4-bit): ~3 GB
  LoRA + optimizer: ~1.5 GB
  Estimated peak: ~19 GB / 80 GB


## Pre-Copy Data to Local SSD

In [4]:
import json, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

LOCAL_TRAIN_DIR = "/content/immunopath_local_v3"
os.makedirs(LOCAL_TRAIN_DIR, exist_ok=True)

print("Pre-copying training data to local SSD...")

# Fallback to training/ if training_v3/ doesn't exist
if not os.path.exists(cfg.train_path):
    print(f"{cfg.train_path} not found, trying training/ directory...")
    TRAINING_DIR_FALLBACK = f"{DATA_DIR}/training"
    cfg.train_path = f"{TRAINING_DIR_FALLBACK}/train.jsonl"
    cfg.val_path = f"{TRAINING_DIR_FALLBACK}/val.jsonl"

# Copy JSONL files
for split in ["train.jsonl", "val.jsonl"]:
    src = cfg.train_path.replace("train.jsonl", split)
    dst = f"{LOCAL_TRAIN_DIR}/{split}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  Copied {split}")

# Collect all patch files
DRIVE_PREFIX = "/content/drive/MyDrive/ImmunoPath"
patch_files = set()
for split in ["train.jsonl", "val.jsonl"]:
    local_path = f"{LOCAL_TRAIN_DIR}/{split}"
    if not os.path.exists(local_path):
        continue
    with open(local_path) as f:
        for line in f:
            sample = json.loads(line.strip())
            for p in sample.get("patch_paths", [])[:cfg.max_patches]:
                if p.startswith(DRIVE_PREFIX):
                    patch_files.add(p)

print(f"  Total patch files referenced: {len(patch_files)}")

# Create directories
dst_dirs = set()
for src_file in patch_files:
    dst_dirs.add(os.path.dirname(src_file.replace(DRIVE_PREFIX, LOCAL_TRAIN_DIR)))
for d in dst_dirs:
    os.makedirs(d, exist_ok=True)

# Copy only missing files
to_copy = []
for src_file in patch_files:
    dst_file = src_file.replace(DRIVE_PREFIX, LOCAL_TRAIN_DIR)
    if not os.path.exists(dst_file):
        to_copy.append((src_file, dst_file))

print(f"  Need to copy: {len(to_copy)} (already cached: {len(patch_files) - len(to_copy)})")

def _copy_one(pair):
    shutil.copy2(pair[0], pair[1])

if to_copy:
    with ThreadPoolExecutor(max_workers=16) as ex:
        list(tqdm(ex.map(_copy_one, to_copy), total=len(to_copy), desc="Copying patches"))

# Rewrite paths in JSONL to point to local SSD
for split in ["train.jsonl", "val.jsonl"]:
    local_path = f"{LOCAL_TRAIN_DIR}/{split}"
    if not os.path.exists(local_path):
        continue
    updated = []
    with open(local_path) as f:
        for line in f:
            sample = json.loads(line.strip())
            sample["patch_paths"] = [
                p.replace(DRIVE_PREFIX, LOCAL_TRAIN_DIR) if p.startswith(DRIVE_PREFIX) else p
                for p in sample["patch_paths"]
            ]
            updated.append(json.dumps(sample))
    with open(local_path, "w") as f:
        f.write("\n".join(updated) + "\n")

# Validate
missing = []
for split in ["train.jsonl", "val.jsonl"]:
    local_path = f"{LOCAL_TRAIN_DIR}/{split}"
    if not os.path.exists(local_path):
        continue
    with open(local_path) as f:
        for line in f:
            sample = json.loads(line.strip())
            for p in sample["patch_paths"][:cfg.max_patches]:
                if not os.path.exists(p):
                    missing.append(p)
if missing:
    print(f"  ⚠ {len(missing)} patches missing")
else:
    print("  All patches verified ✓")

cfg.train_path = f"{LOCAL_TRAIN_DIR}/train.jsonl"
cfg.val_path = f"{LOCAL_TRAIN_DIR}/val.jsonl"
print(f"\nSSD pre-copy complete. Training from {LOCAL_TRAIN_DIR}")

Pre-copying training data to local SSD...
  Copied train.jsonl
  Copied val.jsonl
  Total patch files referenced: 1690
  Need to copy: 1690 (already cached: 0)


Copying patches:   0%|          | 0/1690 [00:00<?, ?it/s]

  All patches verified ✓

SSD pre-copy complete. Training from /content/immunopath_local_v3


## Pre-Cache ALL Images to RAM

In [5]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import time

IMAGE_CACHE = {}

def _load_image(path):
    """Load image at native resolution.

    The processor will resize to 896×896 for SigLIP internally.
    Each image → 4096 SigLIP patches → 256 LLM tokens after projector.
    """
    try:
        img = Image.open(path).convert("RGB")
        img.load()  # Force into RAM, close file handle
        return path, img
    except Exception:
        return path, None

all_patch_paths = set()
for split in ["train.jsonl", "val.jsonl"]:
    local_path = f"{LOCAL_TRAIN_DIR}/{split}"
    if not os.path.exists(local_path):
        continue
    with open(local_path) as f:
        for line in f:
            sample = json.loads(line.strip())
            for p in sample.get("patch_paths", [])[:cfg.max_patches]:
                all_patch_paths.add(p)

print(f"Pre-caching {len(all_patch_paths)} unique patches to RAM...")
cache_start = time.time()

with ThreadPoolExecutor(max_workers=16) as pool:
    results = list(tqdm(
        pool.map(_load_image, all_patch_paths),
        total=len(all_patch_paths),
        desc="Caching images"
    ))

for path, img in results:
    if img is not None:
        IMAGE_CACHE[path] = img

cache_elapsed = time.time() - cache_start
cache_mb = sum(img.size[0] * img.size[1] * 3 for img in IMAGE_CACHE.values()) / 1e6
print(f"\n  Cached {len(IMAGE_CACHE)}/{len(all_patch_paths)} images in {cache_elapsed:.1f}s")
print(f"  RAM usage: ~{cache_mb:.0f} MB")
print(f"  Processor will resize to 896×896 → SigLIP → projector → 256 LLM tokens per image")
print(f"  4 images × 256 = 1024 tokens + ~300 text = ~1324 total (fits in max_length={cfg.max_length})")


Pre-caching 1690 unique patches to RAM...


Caching images:   0%|          | 0/1690 [00:00<?, ?it/s]


  Cached 1690/1690 images in 1.4s
  RAM usage: ~1329 MB
  Processor will resize to 896×896 → SigLIP → projector → 256 LLM tokens per image
  4 images × 256 = 1024 tokens + ~300 text = ~1324 total (fits in max_length=2048)


## Load Model

In [6]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

print(f"Loading {cfg.model_id}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

attn_impl = "flash_attention_2" if FLASH_ATTN_AVAILABLE else "eager"

model = AutoModelForImageTextToText.from_pretrained(
    cfg.model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation=attn_impl,
    quantization_config=bnb_config,
)

processor = AutoProcessor.from_pretrained(cfg.model_id)
processor.tokenizer.padding_side = "right"  # Right padding for training

# === PROCESSOR SETTINGS ===
# do_pan_and_scan is already False by default in Gemma3ProcessorKwargs.
# Explicitly set it to be safe. Do NOT change processor.image_processor.size —
# SigLIP REQUIRES 896×896 input (4096 position embeddings = 64×64 grid, patch_size=14).
# The projector then avg-pools 64×64 → 16×16 = 256 LLM tokens per image.
processor.image_processor.do_pan_and_scan = False
# DO NOT set: processor.image_processor.size = {"longest_edge": 224}
# That would create 16×16=256 patches vs 4096 position embeddings → CRASH

# Verify settings
pan_scan_off = not getattr(processor.image_processor, "do_pan_and_scan", True)
img_size = processor.image_processor.size
print(f"  do_pan_and_scan: {not pan_scan_off} (should be False)")
print(f"  image_processor.size: {img_size} (should be 896)")

# Quick token count check — 1 image should produce 256 LLM tokens
test_img = Image.new("RGB", (512, 512), "white")
test_msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "test"}]}]
test_text = processor.apply_chat_template(test_msgs, add_generation_prompt=True, tokenize=False)
test_inputs = processor(text=[test_text], images=[[test_img]], return_tensors="pt")
n_img_tokens = (test_inputs["input_ids"] == 262144).sum().item()
print(f"  Test: 1 image → {n_img_tokens} LLM tokens (expect 256)")
if n_img_tokens != 256:
    print(f"  ⚠ WARNING: Expected 256 image tokens, got {n_img_tokens}")
else:
    print(f"  ✓ 256 tokens per image confirmed")
del test_inputs, test_img

allocated = torch.cuda.memory_allocated() / 1e9
print(f"\nModel loaded ({attn_impl}), VRAM: {allocated:.2f} GB")
print(f"Tokenizer padding_side: {processor.tokenizer.padding_side}")
print(f"tie_word_embeddings: {model.config.tie_word_embeddings}")


Loading google/medgemma-1.5-4b-it...


config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

  do_pan_and_scan: False (should be False)
  image_processor.size: {'height': 896, 'width': 896} (should be 896)
  Test: 1 image → 256 LLM tokens (expect 256)
  ✓ 256 tokens per image confirmed

Model loaded (eager), VRAM: 3.23 GB
Tokenizer padding_side: right
tie_word_embeddings: True


## LoRA Config (v3: NO modules_to_save — PEFT #1750/#2864)

In [7]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    r=cfg.lora_r,
    bias="none",
    target_modules=cfg.target_modules,
    task_type="CAUSAL_LM",
    # V3 FIX #2: NO modules_to_save
    # PEFT #1750: "Passing both embed_tokens and lm_head to modules_to_save UNTIES them"
    # PEFT #2864: With default ensure_weight_tying=False, both in modules_to_save = "treat as separate"
    # PEFT maintainer: "not add the embeddings to modules_to_save"
    # This corrupted v2's predictions. LoRA alone is sufficient for our task.
)

print(f"LoRA Config (v3):")
print(f"  r={peft_config.r}, alpha={peft_config.lora_alpha}")
print(f"  target_modules: {peft_config.target_modules}")
print(f"  modules_to_save: {peft_config.modules_to_save}  ← REMOVED (v3 fix)")
print(f"  All learning happens through LoRA adapters on internal layers")

LoRA Config (v3):
  r=16, alpha=16
  target_modules: all-linear
  modules_to_save: None  ← REMOVED (v3 fix)
  All learning happens through LoRA adapters on internal layers


## Build Dataset (with MSI-H Oversampling)

In [8]:
import json
from datasets import Dataset
from typing import Any


def load_jsonl(path: str) -> list:
    records = []
    with open(path) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line.strip()))
    return records


def oversample_minority_classes(records: list, msi_factor: int = 5) -> list:
    """V3.1: Oversample minority classes (MSI-H disabled, TME unknown excluded).

    v2 had ~2% MSI-H → model predicted ALL MSS (AUC=0.50, zero discrimination).
    5x oversampling brings MSI-H from ~30 to ~150 samples in 1502-sample dataset.
    """
    msi_h_samples = []
    msi_h_count = 0
    mss_count = 0
    tme_counts = {}

    for r in records:
        try:
            resp = json.loads(r["response"])
            msi = resp.get("msi_status", "").upper()
            tme = resp.get("tme_subtype", "")
            if msi == "MSI-H":
                msi_h_samples.append(r)
                msi_h_count += 1
            elif msi == "MSS":
                mss_count += 1
            if tme:
                tme_counts[tme] = tme_counts.get(tme, 0) + 1
        except Exception:
            pass

    print(f"  Original class distribution:")
    print(f"    MSI-H: {msi_h_count}, MSS: {mss_count}")
    print(f"    TME: {tme_counts}")

    # Oversample MSI-H
    extra = []
    for _ in range(msi_factor - 1):
        extra.extend(msi_h_samples)

    # Also oversample underrepresented TME subtypes
    # V3.1: Exclude TME 'unknown' from oversampling (not in evaluation metrics)
    EVAL_TME_LABELS = {'IE', 'IE/F', 'F', 'D'}
    eval_tme_counts = {k: v for k, v in tme_counts.items() if k in EVAL_TME_LABELS}
    if eval_tme_counts:
        max_tme = max(eval_tme_counts.values())
        for tme_label, count in eval_tme_counts.items():
            if count < max_tme // 3:  # If less than 1/3 of majority
                tme_samples = [
                    r for r in records
                    if json.loads(r["response"]).get("tme_subtype") == tme_label
                ]
                # Oversample to reach 1/3 of majority
                n_extra = (max_tme // 3) - count
                repeats = max(1, n_extra // max(1, len(tme_samples)))
                for _ in range(repeats):
                    extra.extend(tme_samples)
                print(f"    Oversampled TME={tme_label}: +{repeats * len(tme_samples)} samples")

    result = records + extra
    print(f"  After oversampling: {len(result)} samples (+{len(extra)} added)")

    # Verify new distribution
    new_msi_h = sum(1 for r in result if json.loads(r["response"]).get("msi_status", "").upper() == "MSI-H")
    print(f"  New MSI-H count: {new_msi_h} ({100*new_msi_h/len(result):.1f}%)")

    return result


def format_for_sft(example: dict[str, Any]) -> dict[str, Any]:
    n_images = min(len(example.get("patch_paths", [])), cfg.max_patches)
    if n_images == 0:
        n_images = 1

    user_content = [{"type": "image"} for _ in range(n_images)]
    user_content.append({"type": "text", "text": example["prompt"]})

    example["messages"] = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": [{"type": "text", "text": example["response"]}]},
    ]
    return example


# Load raw records
print("Loading datasets...")
train_records = load_jsonl(cfg.train_path)
val_records = load_jsonl(cfg.val_path)
print(f"  Raw train: {len(train_records)}, val: {len(val_records)}")

# V3 FIX #5: Oversample minority classes in training set
print("\nApplying MSI-H oversampling...")
train_records = oversample_minority_classes(train_records, cfg.msi_h_oversample)

# Convert to HuggingFace Dataset
train_data = Dataset.from_list(train_records)
val_data = Dataset.from_list(val_records)

print("\nFormatting for SFT...")
train_data = train_data.map(format_for_sft, desc="Formatting train")
val_data = val_data.map(format_for_sft, desc="Formatting val")

# Shuffle training data (mixes oversampled records)
train_data = train_data.shuffle(seed=42)

print(f"\nFinal datasets:")
print(f"  Train: {len(train_data)} samples")
print(f"  Val:   {len(val_data)} samples")

Loading datasets...
  Raw train: 751, val: 94

Applying MSI-H oversampling...
  Original class distribution:
    MSI-H: 6, MSS: 743
    TME: {'F': 172, 'D': 275, 'IE': 165, 'IE/F': 120, 'unknown': 19}
  After oversampling: 751 samples (+0 added)
  New MSI-H count: 6 (0.8%)

Formatting for SFT...


Formatting train:   0%|          | 0/751 [00:00<?, ? examples/s]

Formatting val:   0%|          | 0/94 [00:00<?, ? examples/s]


Final datasets:
  Train: 751 samples
  Val:   94 samples


## Custom Collator (v3: Response-Only Loss Masking)

**THE critical v3 fix — evidence-backed.**

**Background:** [Google AI Forum](https://discuss.ai.google.dev/t/medgemma-finetuning-padding-and-labels-masking/105422) — Google employee tiffanychen confirmed: *"The notebook follows the default behavior of the SFTTrainer where the model is trained on both the prompt and answers... You could also decide to train on completions only."*

**Evidence:** [QLoRA paper Table 10](https://arxiv.org/abs/2305.14314) — training on completions only improves MMLU by 2-5% across ALL 4 tested datasets (Unnatural Instructions, Alpaca, FLAN v2, Chip2).

**Alternative:** TRL v0.22.0+ supports `assistant_only_loss=True` natively for VLMs. We use a custom collator instead because it also handles image caching and VLM-specific processing.

| | v2 (Google default) | v3 (completions-only) |
|---|---|---|
| Prompt tokens in loss | Yes (~150) | Masked |
| Response tokens in loss | Yes (~200) | Yes (~200) |
| Learning signal focus | Diluted across template | 100% on predictions |

In [9]:
from typing import Any

# Pre-compute the response template token sequence (done ONCE)
_RESPONSE_TEMPLATE_IDS = processor.tokenizer.encode(
    "<start_of_turn>model\n", add_special_tokens=False
)
print(f"Response template token IDs: {_RESPONSE_TEMPLATE_IDS}")
print(f"  Decoded back: '{processor.tokenizer.decode(_RESPONSE_TEMPLATE_IDS)}'")

# Pre-compute special token IDs for masking
_BOI_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids(
    processor.tokenizer.special_tokens_map.get("boi_token", "<start_of_image>")
)
_PAD_TOKEN_ID = processor.tokenizer.pad_token_id
_IMAGE_PLACEHOLDER_ID = 262144

print(f"  pad_token_id: {_PAD_TOKEN_ID}")
print(f"  boi_token_id: {_BOI_TOKEN_ID}")
print(f"  image_placeholder_id: {_IMAGE_PLACEHOLDER_ID}")


def _find_response_start(input_ids_list: list, template_ids: list) -> int:
    """Find the LAST occurrence of the response template in token sequence."""
    tlen = len(template_ids)
    pos = -1
    for i in range(len(input_ids_list) - tlen + 1):
        if input_ids_list[i:i + tlen] == template_ids:
            pos = i + tlen
    return pos


def collate_fn(examples: list[dict[str, Any]]):
    """v3.1 Collator: Response-only loss masking.

    Images are passed at native resolution. The processor resizes to 896×896
    for SigLIP, then the projector avg-pools to 256 LLM tokens per image.
    """
    texts = []
    all_images = []

    for example in examples:
        images = []
        for p in example.get("patch_paths", [])[:cfg.max_patches]:
            if p in IMAGE_CACHE:
                images.append(IMAGE_CACHE[p])
            else:
                try:
                    img = Image.open(p).convert("RGB")
                    images.append(img)
                except Exception:
                    pass
        if not images:
            images = [Image.new("RGB", (512, 512), "white")]

        all_images.append(images)

        text = processor.apply_chat_template(
            example["messages"],
            add_generation_prompt=False,
            tokenize=False,
        ).strip()
        texts.append(text)

    # Joint tokenization — pan_and_scan is OFF on processor object
    batch = processor(
        text=texts,
        images=all_images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=cfg.max_length,
    )

    labels = batch["input_ids"].clone()

    # Mask pad + image tokens
    if _PAD_TOKEN_ID is not None:
        labels[labels == _PAD_TOKEN_ID] = -100
    if _BOI_TOKEN_ID is not None:
        labels[labels == _BOI_TOKEN_ID] = -100
    labels[labels == _IMAGE_PLACEHOLDER_ID] = -100

    # V3 FIX #1: Mask everything BEFORE the response
    for i in range(labels.shape[0]):
        ids_list = batch["input_ids"][i].tolist()
        response_start = _find_response_start(ids_list, _RESPONSE_TEMPLATE_IDS)
        if response_start > 0:
            labels[i, :response_start] = -100

    batch["labels"] = labels
    return batch


# === VERIFY masking + token count ===
print("\n--- Verifying collator ---")
sample = train_data[0]
test_batch = collate_fn([sample])
ids = test_batch["input_ids"][0]
lbls = test_batch["labels"][0]

total_tokens = ids.shape[0]
masked_tokens = (lbls == -100).sum().item()
trained_tokens = total_tokens - masked_tokens
img_tokens = (ids == _IMAGE_PLACEHOLDER_ID).sum().item()

print(f"  Total tokens:   {total_tokens}")
print(f"  Image tokens:   {img_tokens}")
print(f"  Expected:       ~{cfg.max_patches * 256} image tokens (256 per image after projector)")
print(f"  Masked (-100):  {masked_tokens} ({100*masked_tokens/total_tokens:.0f}%)")
print(f"  Trained on:     {trained_tokens} ({100*trained_tokens/total_tokens:.0f}%)")

if total_tokens > cfg.max_length:
    print(f"\n  ⚠ Truncated to max_length={cfg.max_length} — consider increasing if needed")
else:
    print(f"\n  ✓ {total_tokens} tokens fits in max_length={cfg.max_length}")

# Show response text
trained_ids = ids[lbls != -100]
trained_text = processor.tokenizer.decode(trained_ids, skip_special_tokens=True)
print(f"  Response text:  {trained_text[:300]}")

# Show comparison
all_text_tokens = total_tokens - img_tokens
print(f"\n  v2 would train on: ~{all_text_tokens} tokens")
print(f"  v3 trains on:      {trained_tokens} tokens")
if all_text_tokens > 0:
    print(f"  Signal improvement: {all_text_tokens/max(trained_tokens,1):.1f}× more focused")


Response template token IDs: [105, 4368, 107]
  Decoded back: '<start_of_turn>model
'
  pad_token_id: 0
  boi_token_id: 255999
  image_placeholder_id: 262144

--- Verifying collator ---
  Total tokens:   851
  Image tokens:   512
  Expected:       ~512 image tokens (256 per image after projector)
  Masked (-100):  752 (88%)
  Trained on:     99 (12%)

  ✓ 851 tokens fits in max_length=2048
  Response text:  {
  "cd274_expression": "high",
  "msi_status": "MSS",
  "tme_subtype": "IE",
  "til_fraction": 0.611,
  "til_density": "high",
  "immune_phenotype": "inflamed",
  "cd8_infiltration": "high",
  "immune_score": 0.774
}

  v2 would train on: ~339 tokens
  v3 trains on:      99 tokens
  Signal improvement: 3.4× more focused


## Training Configuration + SFTTrainer (Speed-Optimized)

In [10]:
from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback
import time

# Clean previous checkpoints
if os.path.exists(LOCAL_CKPT_DIR):
    existing = [d for d in os.listdir(LOCAL_CKPT_DIR) if d.startswith("checkpoint-")]
    if existing:
        print(f"Cleaning {len(existing)} old checkpoints...")
        for d in existing:
            shutil.rmtree(os.path.join(LOCAL_CKPT_DIR, d), ignore_errors=True)

torch.cuda.empty_cache()
gc.collect()

# Estimate steps
steps_per_epoch = len(train_data) // (cfg.batch_size * cfg.grad_accum)
total_steps = steps_per_epoch * cfg.num_epochs
eval_every = max(steps_per_epoch // 2, 10)
print(f"Steps per epoch: ~{steps_per_epoch}")
print(f"Total steps: ~{total_steps}")
print(f"Eval every {eval_every} steps (~{total_steps // eval_every} evaluations total)")

training_args = SFTConfig(
    output_dir=LOCAL_CKPT_DIR,

    num_train_epochs=cfg.num_epochs,                     # 10
    per_device_train_batch_size=cfg.batch_size,          # 2 (limited by SigLIP VRAM)
    per_device_eval_batch_size=cfg.eval_batch_size,      # 4 (no gradients)
    gradient_accumulation_steps=cfg.grad_accum,          # 8 (effective batch = 16)

    learning_rate=cfg.lr,                                # 1e-4
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    max_grad_norm=cfg.max_grad_norm,
    lr_scheduler_type=cfg.lr_scheduler_type,             # cosine
    optim="adamw_torch_fused",
    bf16=True,

    logging_steps=cfg.logging_steps,
    eval_strategy="steps",
    eval_steps=eval_every,
    save_strategy="steps",
    save_steps=eval_every,
    save_total_limit=3,

    report_to="tensorboard",
    logging_dir=f"{LOCAL_LOG_DIR}/tb_logs",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # gradient_checkpointing=True: recomputes forward pass per segment during backward.
    # Saves significant VRAM. 20% speed cost is worth the memory headroom.
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    label_names=["labels"],

    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=2,
)

early_stopping = EarlyStoppingCallback(early_stopping_patience=2)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=peft_config,
    processing_class=processor,
    data_collator=collate_fn,
    callbacks=[early_stopping],
)

trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in trainer.model.parameters())
allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9

print(f"\nSFTTrainer ready:")
print(f"  Trainable params: {trainable:,} / {total_params:,} ({100*trainable/total_params:.2f}%)")
print(f"  Effective batch: {cfg.batch_size * cfg.grad_accum}")
print(f"  Train samples: {len(train_data)} (with oversampling)")
print(f"  Val samples: {len(val_data)}")
print(f"  Epochs: {cfg.num_epochs}, steps/epoch: ~{steps_per_epoch}")
print(f"  LR: {cfg.lr}, scheduler: {cfg.lr_scheduler_type}")
print(f"  Early stopping patience: {early_stopping.early_stopping_patience} evaluations")
print(f"  ✓ MSI-H oversampling {cfg.msi_h_oversample}×")
print(f"  ✓ Images: native res → 896×896 in SigLIP → 256 LLM tokens/img")
print(f"  ✓ SigLIP VRAM: batch={cfg.batch_size} × {cfg.max_patches} imgs = {cfg.batch_size*cfg.max_patches} through SigLIP")
print(f"  ✓ LLM seq: ~{cfg.max_patches * 256 + 300} tokens")
print(f"  ✓ gradient_checkpointing=True")
flash_str = "FlashAttn-2" if FLASH_ATTN_AVAILABLE else "eager"
print(f"  ✓ Attention: {flash_str}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Steps per epoch: ~46
Total steps: ~138
Eval every 23 steps (~6 evaluations total)

SFTTrainer ready:
  Trainable params: 38,497,792 / 1,623,792,496 (2.37%)
  Effective batch: 16
  Train samples: 751 (with oversampling)
  Val samples: 94
  Epochs: 3, steps/epoch: ~46
  LR: 0.0001, scheduler: cosine
  Early stopping patience: 2 evaluations
  ✓ MSI-H oversampling 1×
  ✓ Images: native res → 896×896 in SigLIP → 256 LLM tokens/img
  ✓ SigLIP VRAM: batch=2 × 2 imgs = 4 through SigLIP
  ✓ LLM seq: ~812 tokens
  ✓ gradient_checkpointing=True
  ✓ Attention: eager


## TRAIN

In [11]:
print("=" * 60)
print("STARTING FINE-TUNING (v3.1)")
print("=" * 60)
print(f"  Model:       {cfg.model_id}")
print(f"  LoRA:        r={cfg.lora_r}, alpha={cfg.lora_alpha}")
print(f"  target:      {cfg.target_modules}")
print(f"  modules_to_save: NONE (PEFT #1750/#2864 fix)")
print(f"  Batch:       {cfg.batch_size}×{cfg.grad_accum}={cfg.batch_size*cfg.grad_accum}")
print(f"  Epochs:      {cfg.num_epochs} + early stopping")
print(f"  LR:          {cfg.lr}")
print(f"  max_length:  {cfg.max_length}")
print(f"  Images:      native → 896×896 → 256 LLM tokens each")
print(f"  Loss mask:   Response-only (QLoRA paper +2-5% MMLU)")
print("=" * 60)

# Reset peak VRAM tracker
torch.cuda.reset_peak_memory_stats()

start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

peak_vram = torch.cuda.max_memory_allocated() / 1e9
current_vram = torch.cuda.memory_allocated() / 1e9

print(f"\n{'=' * 60}")
print(f"TRAINING COMPLETE in {elapsed/60:.1f} minutes")
print(f"Final train loss: {train_result.training_loss:.4f}")
print(f"Peak VRAM: {peak_vram:.1f} GB / 80 GB ({peak_vram/80*100:.0f}% utilization)")
print(f"Current VRAM: {current_vram:.1f} GB")
print(f"{'=' * 60}")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


STARTING FINE-TUNING (v3.1)
  Model:       google/medgemma-1.5-4b-it
  LoRA:        r=16, alpha=16
  target:      all-linear
  modules_to_save: NONE (PEFT #1750/#2864 fix)
  Batch:       2×8=16
  Epochs:      3 + early stopping
  LR:          0.0001
  max_length:  2048
  Images:      native → 896×896 → 256 LLM tokens each
  Loss mask:   Response-only (QLoRA paper +2-5% MMLU)


Step,Training Loss,Validation Loss
23,0.185681,0.164179
46,0.157974,0.155134


Step,Training Loss,Validation Loss
23,0.185681,0.164179
46,0.157974,0.155134
69,0.152384,0.150755
92,0.147706,0.149091
115,0.146020,0.147851
138,0.145575,0.147579



TRAINING COMPLETE in 77.2 minutes
Final train loss: 0.1811
Peak VRAM: 26.2 GB / 80 GB (33% utilization)
Current VRAM: 3.6 GB


## Save LoRA Adapters

In [12]:
# Save locally first (fast), then copy to GDrive
local_adapter_dir = f"{LOCAL_CKPT_DIR}/lora_adapters"
os.makedirs(local_adapter_dir, exist_ok=True)

trainer.save_model(local_adapter_dir)
processor.save_pretrained(local_adapter_dir)
print(f"Adapters saved locally: {local_adapter_dir}")

# Copy to GDrive
print("\nCopying adapters to GDrive...")
gdrive_adapter_dir = f"{cfg.output_dir}/lora_adapters"
os.makedirs(gdrive_adapter_dir, exist_ok=True)

for f_name in os.listdir(local_adapter_dir):
    src = os.path.join(local_adapter_dir, f_name)
    dst = os.path.join(gdrive_adapter_dir, f_name)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
print(f"  → {gdrive_adapter_dir}")

# Also copy to model root
for f_name in os.listdir(local_adapter_dir):
    src = os.path.join(local_adapter_dir, f_name)
    dst = os.path.join(cfg.output_dir, f_name)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
print(f"  → {cfg.output_dir}")

saved_files = os.listdir(gdrive_adapter_dir)
print(f"\nAdapter files: {saved_files}")
print("✓ Adapters saved to both local SSD and GDrive")

Adapters saved locally: /content/immunopath_ckpts_v3/lora_adapters

Copying adapters to GDrive...
  → /content/drive/MyDrive/ImmunoPath/models/immunopath-v3.1/lora_adapters
  → /content/drive/MyDrive/ImmunoPath/models/immunopath-v3.1

Adapter files: ['adapter_model.safetensors', 'adapter_config.json', 'README.md', 'training_args.bin', 'tokenizer_config.json', 'processor_config.json', 'chat_template.jinja', 'tokenizer.json']
✓ Adapters saved to both local SSD and GDrive


## Save Training Log

In [13]:
from datetime import datetime

log_history = trainer.state.log_history
train_logs = [l for l in log_history if "loss" in l and "eval_loss" not in l]
eval_logs = [l for l in log_history if "eval_loss" in l]

training_report = {
    "phase": "5_v3.1",
    "timestamp": datetime.now().isoformat(),
    "model_id": cfg.model_id,
    "v3_fixes": [
        "Response-only loss masking (QLoRA paper: +2-5% MMLU)",
        "REMOVED modules_to_save (PEFT #1750/#2864)",
        f"{cfg.num_epochs} epochs + EarlyStoppingCallback(patience=2)",
        "LR 1e-4 (was 2e-4)",
        "MSI-H oversampling DISABLED (v3.1: 4 patients = memorization)",
        "Images: native res → 896×896 SigLIP → 256 LLM tokens/image",
        f"Batch {cfg.batch_size}×{cfg.grad_accum} = effective {cfg.batch_size*cfg.grad_accum}",
    ],
    "peft_config": {
        "r": cfg.lora_r,
        "alpha": cfg.lora_alpha,
        "dropout": cfg.lora_dropout,
        "target_modules": cfg.target_modules,
        "modules_to_save": "NONE",
    },
    "training_config": {
        "batch_size": cfg.batch_size,
        "grad_accum": cfg.grad_accum,
        "effective_batch": cfg.batch_size * cfg.grad_accum,
        "eval_batch_size": cfg.eval_batch_size,
        "lr": cfg.lr,
        "epochs": cfg.num_epochs,
        "warmup_ratio": cfg.warmup_ratio,
        "max_grad_norm": cfg.max_grad_norm,
        "lr_scheduler": cfg.lr_scheduler_type,
        "optimizer": "adamw_torch_fused",
        "loss_masking": "response-only",
        "early_stopping_patience": 2,
        "image_size": "native (processor resizes to 896x896)",
        "max_length": cfg.max_length,
        "gradient_checkpointing": True,
        "flash_attention": FLASH_ATTN_AVAILABLE,
        "dataloader_num_workers": 4,
    },
    "data": {
        "train_samples": len(train_data),
        "val_samples": len(val_data),
        "msi_h_oversample_factor": cfg.msi_h_oversample,
        "image_cache_size": len(IMAGE_CACHE),
    },
    "results": {
        "final_train_loss": train_result.training_loss,
        "best_eval_loss": min((l["eval_loss"] for l in eval_logs), default=None),
        "training_time_minutes": round(elapsed / 60, 1),
        "total_steps": trainer.state.global_step,
        "stopped_early": trainer.state.global_step < total_steps,
        "peak_vram_gb": round(peak_vram, 1),
        "vram_utilization_pct": round(peak_vram / 80 * 100, 1),
    },
    "train_logs": train_logs,
    "eval_logs": eval_logs,
}

local_log_path = f"{LOCAL_LOG_DIR}/training_log.json"
with open(local_log_path, 'w') as f:
    json.dump(training_report, f, indent=2, default=str)

gdrive_log_path = f"{RESULTS_DIR}/training_log.json"
shutil.copy2(local_log_path, gdrive_log_path)
print(f"Training log saved to {gdrive_log_path}")

best_eval = training_report["results"]["best_eval_loss"]
print(f"\n  Final train loss:  {train_result.training_loss:.4f}")
print(f"  Best eval loss:    {best_eval}")
print(f"  Training time:     {elapsed/60:.1f} min")
print(f"  Total steps:       {trainer.state.global_step}")
print(f"  Stopped early:     {training_report['results']['stopped_early']}")
print(f"  Peak VRAM:         {peak_vram:.1f} GB / 80 GB ({peak_vram/80*100:.0f}%)")
print(f"  Flash Attention:   {'YES' if FLASH_ATTN_AVAILABLE else 'NO (eager)'}")


Training log saved to /content/drive/MyDrive/ImmunoPath/results/phase5_v3/training_log.json

  Final train loss:  0.1811
  Best eval loss:    0.1475788652896881
  Training time:     77.2 min
  Total steps:       141
  Stopped early:     False
  Peak VRAM:         26.2 GB / 80 GB (33%)
  Flash Attention:   NO (eager)


## Quick Inference Test

In [14]:
# Switch to left padding for inference
processor.tokenizer.padding_side = "left"

val_jsonl = f"{LOCAL_TRAIN_DIR}/val.jsonl"
val_record = None
if os.path.exists(val_jsonl):
    with open(val_jsonl) as f:
        val_record = json.loads(f.readline().strip())

if val_record:
    images = []
    for p in val_record["patch_paths"][:cfg.max_patches]:
        if p in IMAGE_CACHE:
            images.append(IMAGE_CACHE[p])
        else:
            try:
                img = Image.open(p).convert("RGB")
                images.append(img)
            except Exception:
                pass
    if not images:
        images = [Image.new("RGB", (512, 512), "white")]

    content = [{"type": "image", "image": img} for img in images]
    content.append({"type": "text", "text": val_record["prompt"]})
    messages = [{"role": "user", "content": content}]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    device = next(p.device for p in trainer.model.parameters() if p.device.type != 'meta')
    inputs = {
        k: v.to(device, dtype=torch.bfloat16) if v.is_floating_point()
        else v.to(device)
        for k, v in inputs.items()
        if torch.is_tensor(v)
    }
    input_len = inputs["input_ids"].shape[-1]

    trainer.model.eval()
    with torch.inference_mode():
        output_ids = trainer.model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            use_cache=True,
        )

    generated = output_ids[0][input_len:]
    response_text = processor.decode(generated, skip_special_tokens=True).strip()

    print("INFERENCE TEST (v3.1):")
    print(f"  Output: {response_text[:400]}")
    print()

    try:
        clean = response_text
        if "```json" in clean:
            clean = clean.split("```json")[1].split("```")[0]
        elif "```" in clean:
            clean = clean.split("```")[1].split("```")[0]
        start = clean.find("{")
        end = clean.rfind("}") + 1
        if start != -1 and end > start:
            parsed = json.loads(clean[start:end])
            print(f"  Valid JSON: {list(parsed.keys())}")
            print(f"  Parsed: {json.dumps(parsed, indent=2)}")
        else:
            print("  No JSON found in output")
    except json.JSONDecodeError as e:
        print(f"  JSON parse error: {e}")

    print(f"\n  Ground truth:")
    gt = json.loads(val_record["response"])
    print(f"  {json.dumps(gt, indent=2)}")
else:
    print("No val samples available")

processor.tokenizer.padding_side = "right"


INFERENCE TEST (v3.1):
  Output: {
  "cd274_expression": "low",
  "msi_status": "MSS",
  "tme_subtype": "D",
  "til_fraction": 0.29,
  "til_density": "low",
  "immune_phenotype": "desert",
  "cd8_infiltration": "low",
  "immune_score": 0.279
}

  Valid JSON: ['cd274_expression', 'msi_status', 'tme_subtype', 'til_fraction', 'til_density', 'immune_phenotype', 'cd8_infiltration', 'immune_score']
  Parsed: {
  "cd274_expression": "low",
  "msi_status": "MSS",
  "tme_subtype": "D",
  "til_fraction": 0.29,
  "til_density": "low",
  "immune_phenotype": "desert",
  "cd8_infiltration": "low",
  "immune_score": 0.279
}

  Ground truth:
  {
  "cd274_expression": "low",
  "msi_status": "MSS",
  "tme_subtype": "F",
  "til_fraction": 0.315,
  "til_density": "moderate",
  "immune_phenotype": "desert",
  "cd8_infiltration": "low",
  "immune_score": 0.335
}


## Phase 5 v3 Summary

In [15]:
print("\n" + "=" * 60)
print("PHASE 5 v3.1 — FINE-TUNING COMPLETE")
print("=" * 60)
print(f"\n  Model:       {cfg.model_id}")
print(f"  PEFT:        LoRA (r={cfg.lora_r}, α={cfg.lora_alpha})")
print(f"  Train:       {len(train_data)} samples × {cfg.num_epochs} epochs")
print(f"  Val:         {len(val_data)} samples")
print(f"  Final loss:  {train_result.training_loss:.4f}")
print(f"  Best eval:   {best_eval}")
print(f"  Time:        {elapsed/60:.1f} min")
print(f"  Steps:       {trainer.state.global_step}")
print(f"  Peak VRAM:   {peak_vram:.1f} GB / 80 GB ({peak_vram/80*100:.0f}%)")
print(f"  Adapters:    {gdrive_adapter_dir}")

print(f"\n  V3.1 FIXES:")
print(f"  1. Response-only loss masking ✓")
print(f"  2. No modules_to_save (PEFT #1750/#2864) ✓")
print(f"  3. {cfg.num_epochs} epochs + early stopping (patience=2) ✓")
print(f"  4. LR 1e-4, cosine scheduler ✓")
print(f"  5. MSI-H oversampling disabled ✓")
print(f"\n  IMAGE HANDLING:")
print(f"  ✓ Native res → processor 896×896 → SigLIP → projector → 256 LLM tokens/img")
print(f"  ✓ Batch: {cfg.batch_size}×{cfg.grad_accum} = {cfg.batch_size*cfg.grad_accum} effective")
print(f"  ✓ Flash Attention: {'YES' if FLASH_ATTN_AVAILABLE else 'NO (eager)'}")
print(f"  ✓ gradient_checkpointing=True")

print(f"\n{'=' * 60}")
print("NEXT: Phase 6 v3.1 — Evaluation")
print(f"  Load from: {gdrive_adapter_dir}")
print(f"{'=' * 60}")



PHASE 5 v3.1 — FINE-TUNING COMPLETE

  Model:       google/medgemma-1.5-4b-it
  PEFT:        LoRA (r=16, α=16)
  Train:       751 samples × 3 epochs
  Val:         94 samples
  Final loss:  0.1811
  Best eval:   0.1475788652896881
  Time:        77.2 min
  Steps:       141
  Peak VRAM:   26.2 GB / 80 GB (33%)
  Adapters:    /content/drive/MyDrive/ImmunoPath/models/immunopath-v3.1/lora_adapters

  V3.1 FIXES:
  1. Response-only loss masking ✓
  2. No modules_to_save (PEFT #1750/#2864) ✓
  3. 3 epochs + early stopping (patience=2) ✓
  4. LR 1e-4, cosine scheduler ✓
  5. MSI-H oversampling disabled ✓

  IMAGE HANDLING:
  ✓ Native res → processor 896×896 → SigLIP → projector → 256 LLM tokens/img
  ✓ Batch: 2×8 = 16 effective
  ✓ Flash Attention: NO (eager)
  ✓ gradient_checkpointing=True

NEXT: Phase 6 v3.1 — Evaluation
  Load from: /content/drive/MyDrive/ImmunoPath/models/immunopath-v3.1/lora_adapters
